In [1]:
#!pip install pandas
#!pip install scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import cross_val_score
import joblib

In [3]:
# Load data
df = pd.read_csv('mushroom_mixed_50000.csv')

In [4]:
display(df)

,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,e,9.39,f,?,n,t,?,NaN,w,9.04,...,b,?,w,u,w,t,g,?,d,u
1,p,15.42,f,k,n,f,d,c,y,6.15,...,?,k,n,?,NaN,f,f,?,d,u
2,p,6.07,s,?,e,t,d,c,b,6.80,...,?,NaN,n,?,NaN,f,f,?,d,a
3,e,4.64,x,t,n,f,a,d,y,8.37,...,?,k,n,?,NaN,f,f,?,d,s
4,p,17.87,f,h,e,f,e,?,w,19.03,...,s,y,w,u,w,t,g,?,d,u
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,p,14.95,x,s,n,t,p,?,e,9.74,...,c,?,y,?,NaN,f,f,?,d,a
49996,e,3.40,b,g,w,f,x,d,w,4.09,...,?,NaN,y,?,NaN,f,f,?,g,a
49997,e,11.53,x,?,n,t,?,NaN,w,10.32,...,b,?,w,u,w,t,g,?,d,u
49998,p,3.14,f,d,b,f,x,c,w,4.62,...,b,i,n,?,NaN,f,f,?,l,a


In [5]:
df.shape

(50000, 21)

In [6]:
df.dtypes

class                    object
cap-diameter            float64
cap-shape                object
cap-surface              object
cap-color                object
does-bruise-or-bleed     object
gill-attachment          object
gill-spacing             object
gill-color               object
stem-height             float64
stem-width              float64
stem-root                object
stem-surface             object
stem-color               object
veil-type                object
veil-color               object
has-ring                 object
ring-type                object
spore-print-color        object
habitat                  object
season                   object
dtype: object

In [7]:
df = df.drop_duplicates()

display(df)

,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,e,9.39,f,?,n,t,?,NaN,w,9.04,...,b,?,w,u,w,t,g,?,d,u
1,p,15.42,f,k,n,f,d,c,y,6.15,...,?,k,n,?,NaN,f,f,?,d,u
2,p,6.07,s,?,e,t,d,c,b,6.80,...,?,NaN,n,?,NaN,f,f,?,d,a
3,e,4.64,x,t,n,f,a,d,y,8.37,...,?,k,n,?,NaN,f,f,?,d,s
4,p,17.87,f,h,e,f,e,?,w,19.03,...,s,y,w,u,w,t,g,?,d,u
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,p,14.95,x,s,n,t,p,?,e,9.74,...,c,?,y,?,NaN,f,f,?,d,a
49996,e,3.40,b,g,w,f,x,d,w,4.09,...,?,NaN,y,?,NaN,f,f,?,g,a
49997,e,11.53,x,?,n,t,?,NaN,w,10.32,...,b,?,w,u,w,t,g,?,d,u
49998,p,3.14,f,d,b,f,x,c,w,4.62,...,b,i,n,?,NaN,f,f,?,l,a


In [8]:
df.replace("?", np.nan, inplace=True)

In [9]:
df.isnull().sum()

class                       0
cap-diameter                0
cap-shape                   0
cap-surface             11580
cap-color                   0
does-bruise-or-bleed        0
gill-attachment          8071
gill-spacing            20496
gill-color                  0
stem-height                 0
stem-width                  0
stem-root               42171
stem-surface            31204
stem-color                  0
veil-type               47292
veil-color              43842
has-ring                    0
ring-type                2029
spore-print-color       44708
habitat                     0
season                      0
dtype: int64

In [10]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
display(missing_percentage)

class                    0.000000
cap-diameter             0.000000
cap-shape                0.000000
cap-surface             23.207343
cap-color                0.000000
does-bruise-or-bleed     0.000000
gill-attachment         16.174997
gill-spacing            41.075795
gill-color               0.000000
stem-height              0.000000
stem-width               0.000000
stem-root               84.514409
stem-surface            62.535573
stem-color               0.000000
veil-type               94.777346
veil-color              87.863241
has-ring                 0.000000
ring-type                4.066295
spore-print-color       89.598782
habitat                  0.000000
season                   0.000000
dtype: float64

In [11]:
threshold = 85
# Select columns to drop
columns_to_drop = missing_percentage[missing_percentage > threshold].index

# Drop the columns
df.drop(columns=columns_to_drop, inplace=True)

# Display remaining columns
print("Dropped columns:", columns_to_drop)
print("Remaining columns:", df.columns)

Dropped columns: Index(['veil-type', 'veil-color', 'spore-print-color'], dtype='object')
Remaining columns: Index(['class', 'cap-diameter', 'cap-shape', 'cap-surface', 'cap-color',
       'does-bruise-or-bleed', 'gill-attachment', 'gill-spacing', 'gill-color',
       'stem-height', 'stem-width', 'stem-root', 'stem-surface', 'stem-color',
       'has-ring', 'ring-type', 'habitat', 'season'],
      dtype='object')


In [12]:
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [13]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [14]:
#joblib.dump(label_encoders, "label_encoders.pkl")

In [15]:
display(df)

,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,stem-width,stem-root,stem-surface,stem-color,has-ring,ring-type,habitat,season
0,0,9.39,2,11,5,1,7,3,10,9.04,16.26,0,8,11,1,2,0,2
1,1,15.42,2,5,5,0,1,0,11,6.15,32.78,5,4,6,0,1,0,2
2,1,6.07,5,11,1,1,1,0,0,6.80,6.53,5,8,6,0,1,0,0
3,0,4.64,6,8,5,0,0,1,11,8.37,6.52,5,4,6,0,1,0,1
4,1,17.87,2,3,1,0,2,3,10,19.03,18.39,4,7,11,1,2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,1,14.95,6,7,5,1,4,3,1,9.74,37.02,1,8,12,0,1,0,0
49996,0,3.40,0,2,10,0,6,1,10,4.09,2.86,5,8,12,0,1,1,0
49997,0,11.53,6,11,5,1,7,3,10,10.32,16.41,0,8,11,1,2,0,2
49998,1,3.14,2,0,0,0,6,0,10,4.62,3.24,0,3,6,0,1,3,0


In [16]:
x = df.iloc[:, df.columns != 'class']
y = df.iloc[:, df.columns == 'class']

In [17]:
display(x)

,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,stem-width,stem-root,stem-surface,stem-color,has-ring,ring-type,habitat,season
0,9.39,2,11,5,1,7,3,10,9.04,16.26,0,8,11,1,2,0,2
1,15.42,2,5,5,0,1,0,11,6.15,32.78,5,4,6,0,1,0,2
2,6.07,5,11,1,1,1,0,0,6.80,6.53,5,8,6,0,1,0,0
3,4.64,6,8,5,0,0,1,11,8.37,6.52,5,4,6,0,1,0,1
4,17.87,2,3,1,0,2,3,10,19.03,18.39,4,7,11,1,2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,14.95,6,7,5,1,4,3,1,9.74,37.02,1,8,12,0,1,0,0
49996,3.40,0,2,10,0,6,1,10,4.09,2.86,5,8,12,0,1,1,0
49997,11.53,6,11,5,1,7,3,10,10.32,16.41,0,8,11,1,2,0,2
49998,3.14,2,0,0,0,6,0,10,4.62,3.24,0,3,6,0,1,3,0


In [18]:
display(y)

,class
0,0
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,1


In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [20]:
display(X_train)

,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,stem-width,stem-root,stem-surface,stem-color,has-ring,ring-type,habitat,season
3981,2.85,2,9,6,1,1,0,11,3.96,6.60,5,8,6,0,1,0,0
7094,6.59,2,10,11,0,5,0,11,6.52,16.63,5,7,12,0,1,0,0
16977,7.45,6,2,5,0,2,3,10,8.56,14.67,5,7,11,1,5,0,2
34245,1.93,1,8,11,0,7,3,11,7.31,3.11,5,8,12,0,1,1,0
40502,4.63,6,3,5,0,0,0,5,4.41,6.72,5,7,7,1,7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11296,1.64,0,3,5,0,0,3,5,5.96,1.74,5,6,6,0,1,0,2
44820,9.23,6,1,5,0,0,3,5,10.24,15.61,4,8,6,0,1,0,0
38218,10.90,6,8,5,0,0,1,10,12.39,34.66,5,8,11,0,1,0,2
860,19.69,2,5,5,0,1,0,11,7.04,44.84,5,4,6,0,1,0,2


In [21]:
columns = x.columns
print(columns)

Index(['cap-diameter', 'cap-shape', 'cap-surface', 'cap-color',
       'does-bruise-or-bleed', 'gill-attachment', 'gill-spacing', 'gill-color',
       'stem-height', 'stem-width', 'stem-root', 'stem-surface', 'stem-color',
       'has-ring', 'ring-type', 'habitat', 'season'],
      dtype='object')


In [22]:
imputer = SimpleImputer(missing_values=np.nan, strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test) 

In [23]:
X_train_df = pd.DataFrame(X_train, columns=columns)
X_test_df = pd.DataFrame(X_test, columns=columns)

In [24]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [25]:
Y_train = Y_train.values.ravel()
Y_test = Y_test.values.ravel()

In [26]:
k = X_train.shape[1]
selector = SelectKBest(score_func=mutual_info_classif, k=k)
X_train = selector.fit_transform(X_train_df, Y_train)
X_test = selector.transform(X_test_df)

In [27]:
selected_features = X_train_df.columns[selector.get_support()]
joblib.dump(selected_features, "selected_features.pkl")
joblib.dump(selector, "feature_selector.pkl")

['feature_selector.pkl']

In [28]:
print(selected_features)

Index(['cap-diameter', 'cap-shape', 'cap-surface', 'cap-color',
       'does-bruise-or-bleed', 'gill-attachment', 'gill-spacing', 'gill-color',
       'stem-height', 'stem-width', 'stem-root', 'stem-surface', 'stem-color',
       'has-ring', 'ring-type', 'habitat', 'season'],
      dtype='object')


In [29]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, Y_train)
dt_pred = dt_model.predict(X_test)
dt_accuracy = accuracy_score(Y_test, dt_pred)*100
print(f"Decision Tree Accuracy: {dt_accuracy:.2f}","%")

Decision Tree Accuracy: 99.73 %


In [30]:
dt_params = {
    'max_depth': [10, 20, None], 
    'min_samples_split': [5, 10], 
    'min_samples_leaf': [2, 4], 
    'ccp_alpha': [0.01, 0.1],
    'max_features': ['sqrt', 'log2', None],
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, scoring='accuracy', n_jobs=-1)
dt_grid.fit(X_train, Y_train)
best_dt = dt_grid.best_estimator_
best_dt_pred = best_dt.predict(X_test)
best_dt_accuracy = accuracy_score(Y_test, best_dt_pred)*100
print(f"Decision Tree Accuracy after GridSearchCV: {best_dt_accuracy:.2f}","%")
print(f"Best Decision Tree Parameters: {dt_grid.best_params_}")


Decision Tree Accuracy after GridSearchCV: 78.83 %
Best Decision Tree Parameters: {'ccp_alpha': 0.01, 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 5}


In [31]:
joblib.dump(best_dt, 'decision_tree_model.pkl')

['decision_tree_model.pkl']

In [32]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, Y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(Y_test, rf_pred)*100
print(f"Random Forest Accuracy: {rf_accuracy:.2f}","%")

Random Forest Accuracy: 99.99 %


In [33]:
rf_params = {
    'n_estimators': [50, 100],  
    'max_depth': [10, 20],  
    'min_samples_split': [5, 10],  
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, Y_train)
best_rf = rf_grid.best_estimator_
best_rf_pred = best_rf.predict(X_test)
best_rf_accuracy = accuracy_score(Y_test, best_rf_pred)*100
print(f"Random Forest Accuracy after GridSearchCV: {best_rf_accuracy:.2f}","%")
print(f"Best Random Forest Parameters: {rf_grid.best_params_}")

Random Forest Accuracy after GridSearchCV: 99.98 %
Best Random Forest Parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}


In [34]:
joblib.dump(best_rf, 'random_forest_model.pkl')

['random_forest_model.pkl']

In [35]:
knn_model = KNeighborsClassifier()
knn_model.fit(X_train, Y_train)
knn_pred = knn_model.predict(X_test)
knn_accuracy = accuracy_score(Y_test, knn_pred)*100
print(f"KNN Accuracy before GridSearchCV: {knn_accuracy:.2f}","%")

KNN Accuracy before GridSearchCV: 99.90 %


In [36]:
knn_params = {
    'n_neighbors': [5, 7, 9],  
    'weights': ['uniform', 'distance'],  
    'metric': ['euclidean']  
}

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train, Y_train)
best_knn = knn_grid.best_estimator_
best_knn_pred = best_knn.predict(X_test)
best_knn_accuracy = accuracy_score(Y_test, best_knn_pred)*100
print(f"KNN Accuracy after GridSearchCV: {best_knn_accuracy:.2f}","%")
print(f"Best KNN Parameters: {knn_grid.best_params_}")

KNN Accuracy after GridSearchCV: 99.91 %
Best KNN Parameters: {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'distance'}


In [37]:
joblib.dump(best_knn, 'knn_model.pkl')

['knn_model.pkl']

In [38]:
with open("decision_tree_model.pkl", "rb") as file:
    decision_tree_model = joblib.load(file)

with open("random_forest_model.pkl", "rb") as file:
    random_forest_model = joblib.load(file)

with open("knn_model.pkl", "rb") as file:
    knn_model = joblib.load(file)

In [39]:
models = {
    "Model_1": decision_tree_model,  # Replace with actual model names
    "Model_2": random_forest_model,
    "Model_3": knn_model
}

In [40]:
for name, model in models.items():
    cv_scores = cross_val_score(model, x, y.values.ravel(), cv=5, scoring='accuracy')
    mean_accuracy = np.mean(cv_scores)
    std_accuracy = np.std(cv_scores)
    
    print(f"{name} - Cross-Validation Accuracy Scores: {cv_scores}")
    print(f"{name} - Mean Accuracy: {mean_accuracy:.4f}")
    print(f"{name} - Standard Deviation of Accuracy: {std_accuracy:.4f}")
    print("-" * 50)

Model_1 - Cross-Validation Accuracy Scores: [0.74599198 0.74418838 0.751002   0.73865117 0.78955807]
Model_1 - Mean Accuracy: 0.7539
Model_1 - Standard Deviation of Accuracy: 0.0183
--------------------------------------------------
Model_2 - Cross-Validation Accuracy Scores: [0.9998998  0.9996994  0.9998998  0.99929853 1.        ]
Model_2 - Mean Accuracy: 0.9998
Model_2 - Standard Deviation of Accuracy: 0.0003
--------------------------------------------------
Model_3 - Cross-Validation Accuracy Scores: [0.999499   0.998998   0.9988978  0.99929853 0.99919832]
Model_3 - Mean Accuracy: 0.9992
Model_3 - Standard Deviation of Accuracy: 0.0002
--------------------------------------------------


In [43]:
joblib.dump(best_rf, 'proj1_chosen_model.pkl')

['proj1_chosen_model.pkl']